# Experiment v6 — Calibration + Isolation Phase (Jupyter server)

Run the cells in order. **Section 3 (C0 calibration) comes before the main run and its output
decides the budget levels used in section 4** — see `analysis/EXPERIMENT_V6_PLAN.md`.

What is new in v6:
- **Greedy initial sequence** (standard EMCTS's default), shared across all 12 configs per (scenario, visit).
- **The budget bounds SEARCH only** — greedy initialization is measured and excluded; bridge burning,
  including the crossover tournament, stays inside the budget.
- **50 visits** per scenario (v5: 100).
- **Realized spend logged per turn** (units, advances, copies, evals, init cost, wall-clock).

In [ ]:
# EDIT to the absolute path of the Tribes-master directory on the server:
TRIBES = '/root/Bachelor_Thesis/Tribes-master'
%cd $TRIBES

## 0. Environment check
Java version, cores, free RAM and disk. The C0 calibration needs a **quiesced** machine — if anything
else is running, its numbers are not reference-machine numbers.

In [ ]:
!java -version 2>&1 | head -3
!echo "cores : $(nproc)"
!free -g | head -2
!df -h . | tail -1
print('--- other java processes (should be none before C0) ---')
!pgrep -af java || echo '(none - good)'

## 1. Build
Recompiles everything from `src/`. The source list is regenerated each time, so new v6 classes are
picked up automatically.

In [ ]:
!rm -rf out && mkdir -p out
!find src -name '*.java' > _srcs.txt
!javac -d out -cp "lib/*" @_srcs.txt && echo 'BUILD OK' || echo 'BUILD FAILED'

## 2. Smoke test (~30 s) — pipeline correctness
Verifies FastGameStateIO round-trip, the shared **greedy** initial sequence across agents AND budgets,
folder layout, CSV schema, operator telemetry, and — new in v6 — the two budget-boundary properties:

- **init OUT of the budget**: every agent reports a non-zero initialization cost while its search spend
  still reaches the full budget (if init were charged, search would fall short by exactly that amount);
- **bridge burning IN the budget**: all agents perform 20 burns, the crossover agents stay at the budget
  rather than exceeding it, and they buy strictly fewer distinct sequences than EMCTS — the tournaments
  are paid for out of the same pot.

If this fails, do NOT launch C0 or the full run.

In [ ]:
!java -Xmx4g -cp "out:lib/*" core.data.exp.NewExperimentSmokeTestV6

## 3. C0 — thinking-time calibration (~2.6 h, SINGLE-THREADED)
**Run this before section 4 and read its output.** Gives each of the 12 configs a fixed amount of
SEARCH time (500 / 1000 / 5000 ms, each a separate run) and records what it actually does: advances,
copies, evals, weighted units, and the greedy-initialization cost separately.

It answers four questions:
1. the units→seconds conversion the paper reports for this reference machine;
2. whether that conversion is **agent-invariant** — which decides whether v6 uses one shared
   compute-unit budget or per-agent budgets calibrated to reference time;
3. whether it is invariant across scenario source and branching factor (the H5 confound check);
4. how large `InitUnits` actually is, and therefore whether the smallest budget level needs raising.

The portfolio must exist first; the cell builds it if needed. Leave the machine otherwise idle.

In [ ]:
# Build the v6 portfolio if it is not already there (skipped when complete).
!java -Xmx64g -cp "out:lib/*" core.data.exp.BuildV6Portfolio test_results/v6/_portfolio $(nproc)

In [ ]:
# args: [portfolioDir] [outDir] [targetsMsCsv] [perSource] [reps] [warmup]
!nohup java -Xmx32g -cp "out:lib/*" core.data.exp.CalibrationRunnerV6 \
    test_results/v6/_portfolio test_results/v6/_calibration 500,1000,5000 12 5 40 \
    > v6_calibration.log 2>&1 &
print('C0 started in background -> v6_calibration.log')

In [ ]:
# C0 progress + the per-config summary once it finishes (re-run to refresh).
!echo "rows so far : $(($(wc -l < test_results/v6/_calibration/calibration.csv 2>/dev/null || echo 1)-1))"
!pgrep -af CalibrationRunnerV6 || echo '(C0 not running - finished or not started)'
!tail -n 40 v6_calibration.log 2>/dev/null || echo '(no log yet)'

### 3b. Read C0 → set the v6 budget levels
The table below is the decision. Read **units/ms across configs at a fixed target**: if it is flat, one
shared compute-unit budget is fair and you keep `RunExperimentV6.DEFAULT_COMPUTE_LEVELS`; if it varies
systematically by agent, budgets must be calibrated per agent instead. Also check `InitUnits` against
the smallest level — if initialization plus 20 burns dominates it, raise that level.

In [ ]:
import pandas as pd
c = pd.read_csv('test_results/v6/_calibration/calibration.csv')
c['Config'] = c.Config + c.Ratio.astype(str).radd('@r').where(c.Ratio != 'NA', '')
c['UnitsPerMs'] = c.SearchUnits / c.SearchMillis.clip(lower=1)

print('=== units/ms by config x target (is it flat across configs?) ===')
display(c.pivot_table(index='Config', columns='TargetMillis', values='UnitsPerMs', aggfunc='mean').round(2))

print('=== units/ms by scenario source (the H5 confound check) ===')
display(c.pivot_table(index='Source', columns='TargetMillis', values='UnitsPerMs', aggfunc='mean').round(2))

print('=== greedy-init cost by source, and as a share of the smallest planned level ===')
init = c.groupby('Source').InitUnits.agg(['mean', 'median', 'max']).round(0)
init['share_of_13400'] = (init['mean'] / 13400).round(3)
display(init)

print('=== stability across repetitions (CV should be small on a quiesced machine) ===')
cv = c.groupby(['TargetMillis', 'Config']).SearchUnits.agg(['mean', 'std'])
cv['cv'] = (cv['std'] / cv['mean']).round(4)
display(cv.cv.groupby('TargetMillis').describe()[['mean', 'max']].round(4))

print('=== suggested per-agent budgets, if the per-agent route is taken ===')
display(c.pivot_table(index='Config', columns='TargetMillis', values='SearchUnits', aggfunc='mean').round(0))

## 4. Run v6 (background + log) — also the RESUME cell
Launches the full run detached so it survives a disconnect. **Re-run this same cell to resume** after a
server restart — finished work is skipped, an interrupted config continues per (scenario, visit).

Args: `[threads|auto] [computeUnitsCsv]`. `auto` (default) runs the pre-run phases: portfolio build →
startup verify → **greedy-init invariance check** → **thread auto-tune** → isolation.

**Set the budget levels from C0** by passing them explicitly, e.g. `auto 13400,26800,134000`.

Scale: 200 scenarios × 50 visits × 12 configs × 3 levels = 360,000 turns (half of v5).

In [ ]:
!nohup java -Xmx750g -cp "out:lib/*" core.data.exp.RunExperimentV6 auto > v6_run.log 2>&1 &
print('v6 started/resumed in background -> v6_run.log')

## 5. Progress (re-run to refresh)
Read-only view of the running job: pre-run checkpoints, leaves finished, turns/min with a projected
finish, live thread/CPU use (`NLWP` = OS threads, `%CPU` = total across them — should sit well above
100%), and the log tail.

In [ ]:
import subprocess, time, os, glob

BASE, LEAVES, TURNS_PER_LEAF = 'test_results/v6', 36, 200 * 50

print('cores       :', subprocess.run(['nproc'], capture_output=True, text=True).stdout.strip())
!echo "portfolio   : $(test -f test_results/v6/_portfolio/_portfolio_complete.txt && echo COMPLETE || echo building/incomplete)"
!echo "pre-run     : greedy-init=$(test -f test_results/v6/_greedy_init_ok.txt && echo ok || echo pending)  tuned-threads=$(cat test_results/v6/_tuned_threads.txt 2>/dev/null || echo pending)"
!echo "calibration : $(test -f test_results/v6/_calibration/calibration.csv && echo present || echo MISSING - run section 3 first)"
!echo "leaf CSVs   : $(find test_results/v6 -name results.csv 2>/dev/null | wc -l) / 36 configs"
!echo "configs done: $(grep -c . test_results/v6/_completed.txt 2>/dev/null || echo 0) / 36"

# Turn counts per leaf, from the telemetry file (one row per completed turn).
teles = sorted(glob.glob(f'{BASE}/**/turn_telemetry.csv', recursive=True))
total_turns = 0
print('--- per-leaf completed turns (full leaf = 10000) ---')
for f in teles:
    n = max(0, sum(1 for _ in open(f)) - 1)
    total_turns += n
    if n < TURNS_PER_LEAF:
        print(f'  {f.replace(BASE + "/", ""):50s} {n:6d}')
print(f'  TOTAL {total_turns} / {LEAVES * TURNS_PER_LEAF} turns '
      f'({100.0 * total_turns / (LEAVES * TURNS_PER_LEAF):.1f}%)')

# Throughput + ETA from the log's mtime span.
log = 'v6_run.log'
if os.path.exists(log) and total_turns > 0:
    mins = max(1e-9, (time.time() - os.path.getctime(log)) / 60.0)
    rate = total_turns / mins
    left = LEAVES * TURNS_PER_LEAF - total_turns
    print(f'  rate  {rate:.0f} turns/min   elapsed {mins/60:.1f} h   '
          f'ETA {left / rate / 60:.1f} h' if rate > 0 else '  rate  n/a')

print('--- run process: PID / threads (NLWP) / CPU% / elapsed ---')
!ps -o pid,nlwp,pcpu,etime,comm -C java 2>/dev/null || echo '(no java process running)'
!pgrep -af RunExperimentV6 || echo '(RunExperimentV6 not running - finished, not started, or between phases)'
print('--- last 30 log lines ---')
!tail -n 30 v6_run.log 2>/dev/null || echo '(no log yet)'

## 6. Analyse (after the run completes)
`analyze_v6_full.py` carries over every v5 output so v5→v6 is like-for-like (fig3–fig8, E1–E16, and the
hypothesis battery E17–E21), plus the new budget-adherence section:

| table | content |
|---|---|
| `B1_budget_adherence` | realized `SearchUnits` vs target per agent per level: mean, SD, overshoot |
| `B2_op_mix` | advances/copies/evals per turn, and what equal-fmcalls *would* have given each agent |
| `B3_reference_conversion` | the C0 units→seconds table, with machine provenance |
| `B4_init_cost` | `InitUnits` by source and BF tertile, and its share of the budget |

*(The analysis script is not part of this branch — write it once the run has produced data.)*

In [ ]:
# !python3 analysis/analyze_v6_full.py